In [54]:
import json
import re
import pandas as pd
from pathlib import Path

In [55]:
def load_raw_data(path="raw_data/abcd_v1.1.json"):
    with open(path) as f:
        return json.load(f)

data = load_raw_data()
print({split: len(chats) for split, chats in data.items()})

{'train': 8034, 'dev': 1004, 'test': 1004}


In [56]:
OPENER_PATTERNS = re.compile(
    r"how (can|may|could|do) i (help|assist)"
    r"|what (can|may) i (do for you|help)"
    r"|how can i be of (help|assistance)"
    r"|^\s*(hi|hello|hey)( there)?[\s.,!]*$",
    re.IGNORECASE,
)

def is_opener(text, max_words=20):
    return len(text.split()) <= max_words and bool(OPENER_PATTERNS.search(text))

In [57]:
CHAT_STOPWORDS = {"hi", "hello", "hey", "thanks", "thank", "you", "ok", "okay", "sure", "please"}

def has_enough_semantic_info(text: str, min_words: int = 3) -> bool:
    clean_text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    tokens = clean_text.split()
    meaningful_tokens = [w for w in tokens if w not in CHAT_STOPWORDS]
    return len(meaningful_tokens) >= min_words

def extract_first_messages(turns, stop_on_action=True):
    parts = []
    for speaker, text in turns:
        if speaker == "customer":
            parts.append(text)
        elif speaker == "agent":
            if is_opener(text):
                continue

            current_extracted_text = " ".join(parts)
            if parts and has_enough_semantic_info(current_extracted_text):
                break
            if parts:
                break
        elif speaker == "action":
            if parts and stop_on_action:
                break
    return re.sub(r"\s+", " ", " ".join(parts)).strip()

In [58]:
REFUND_TEXT = "refund has been made"

def has_refund(turns):
    return int(any(t[0] == "action" and REFUND_TEXT in t[1].lower() for t in turns))

rows = []
target_subflows = ("refund_initiate",
    "refund_update",
    "cost",
    "manage_dispute_bill",
    "manage_cancel",
    "status_service_added",
    "status_quantity",
    "return_size",
    "return_color",
    "return_stain",
    "bad_price_competitor",
    "status_mystery_fee", "manage")

for split in ["train", "dev", "test"]:
    for chat in data[split]:
        turns = chat["original"]
        subflow = chat.get("scenario", {}).get("subflow", "")
        label = 1 if subflow in target_subflows else 0
        rows.append({
            "split": split,
            "id": chat["convo_id"],
            "flow": chat["scenario"].get("flow"),
            "subflow": chat["scenario"].get("subflow"),
            "first_msgs": extract_first_messages(turns),
            "refund": has_refund(turns),
            'refund_label': label
        })

abcd = pd.DataFrame(rows)
abcd["n_words"] = abcd["first_msgs"].str.split().str.len()
abcd.shape

(10042, 8)

In [59]:
abcd.head()

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me ...",0,1,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to kn...,0,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deat...",0,1,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. A...,0,0,34


In [ ]:
import re
from typing import List, Tuple

DOMAIN_KEYWORDS = {
    "refund", "refunded", "refunding",
    "return", "returned", "returning",
    "size", "color", "damaged", "defective", "broken",
    "exchange", "cancel", "canceled", "cancelling",
    "wrong", "replace", "replacement",
    "status", "expire", "expiration", "promo", "code"
}

OPENER_PATTERNS = re.compile(
    r"how (can|may|could|do) i (help|assist)"
    r"|what (can|may) i (do for you|help)"
    r"|how can i be of (help|assistance)"
    r"|^\s*(hi|hello|hey)( there)?[\s.,!]*$",
    re.IGNORECASE,
)

CHAT_STOPWORDS = {"hi", "hello", "hey", "thanks", "thank", "you", "ok", "okay", "sure", "please"}

def is_opener(text: str) -> bool:
    return bool(OPENER_PATTERNS.search(text.strip()))

def has_enough_semantic_info(text: str, min_words: int = 3) -> bool:
    clean_text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    tokens = clean_text.split()
    meaningful_tokens = [w for w in tokens if w not in CHAT_STOPWORDS]
    return len(meaningful_tokens) >= min_words


def extract_dual_path_features(
    turns: List[Tuple[str, str]],
    stop_on_action: bool = True,
    first_weight: int = 1
) -> str:
    first_parts = []
    split_index = len(turns)

    for idx, (speaker, text) in enumerate(turns):
        if speaker == "customer":
            first_parts.append(text)
        elif speaker == "agent":
            if is_opener(text):
                continue
            current_extracted = " ".join(first_parts)
            if first_parts and has_enough_semantic_info(current_extracted):
                split_index = idx
                break
        elif speaker == "action":
            current_extracted = " ".join(first_parts)
            if first_parts and stop_on_action and has_enough_semantic_info(current_extracted):
                split_index = idx
                break

    first_intent_text = re.sub(r"\s+", " ", " ".join(first_parts)).strip().lower()

    later_keywords = []

    for speaker, text in turns[split_index:]:
        if speaker == "customer":
            clean_text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
            words = clean_text.split()
            matched = [w for w in words if w in DOMAIN_KEYWORDS]
            if matched:
                later_keywords.extend(matched)

    later_keywords_text = " ".join(later_keywords)

    weighted_first = " ".join([first_intent_text] * first_weight)

    final_input = f"{weighted_first} {later_keywords_text}".strip()
    return re.sub(r"\s+", " ", final_input)

In [61]:
REFUND_TEXT = "refund has been made"

def has_refund(turns):
    return int(any(t[0] == "action" and REFUND_TEXT in t[1].lower() for t in turns))

rows_v2 = []

target_subflows = ("refund_initiate",
    "refund_update",
    "cost",
    "manage_dispute_bill",
    "manage_cancel",
    "status_service_added",
    "status_quantity",
    "return_size",
    "return_color",
    "return_stain",
    "bad_price_competitor",
    "status_mystery_fee", "manage",'competitors','return')

for split in ["train", "dev", "test"]:
    for chat in data[split]:
        turns = chat["original"]
        subflow = chat.get("scenario", {}).get("subflow", "")
        label = 1 if subflow in target_subflows else 0
        rows_v2.append({
            "split": split,
            "id": chat["convo_id"],
            "flow": chat["scenario"].get("flow"),
            "subflow": chat["scenario"].get("subflow"),
            "first_msgs": extract_dual_path_features(turns),
            "refund": has_refund(turns),
            'refund_label': label
        })

abcd_v2 = pd.DataFrame(rows_v2)
abcd_v2["n_words"] = abcd_v2["first_msgs"].str.split().str.len()
abcd_v2.shape

(10042, 8)

In [62]:
#abcd.first_msgs[2]
abcd['first_msgs'].isna().sum()

np.int64(0)

In [63]:
abcd_v2['first_msgs'].isna().sum()

np.int64(0)

In [64]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
abcd.to_parquet("data/processed/abcd_first_msgs.parquet", index=False)

In [65]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
abcd_v2.to_parquet("data/processed/abcd_first_msgs.parquet_v2", index=False)